# GM-CDLD discovery and frozen reuse
Main manuscript: 32D, 20 cycles, 20 reader epochs, seeds 17/43/97. This calls the original training implementation, not a reimplementation. One NBA cycle presents each row 10 times; tennis 4. Initial comparison is not part of this workflow.

Set the prepared input path before running. Full training can take hours. No training output has been prefilled.

In [ ]:
from pathlib import Path
import json, subprocess, sys, shutil
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
assert (ROOT/'code/train.py').is_file(), 'Open from repository root or notebooks/'
print('Repository:', ROOT)


In [ ]:
DATA = ROOT/'work/prepared/data'  # provide NBA/ and tennis/ here
RUN = ROOT/'work/manuscript_run'  # must be a NEW directory
DOMAINS = ['NBA','tennis']
CONDITIONS = ['direct']  # GM; optionally add 'C39' for the supplementary readout variant
SEEDS = [17,43,97]

In [ ]:
subprocess.run([sys.executable,str(ROOT/'tools/stage_run.py'),'--data',str(DATA),'--output',str(RUN)],check=True)
cfg=json.loads((RUN/'plan.json').read_text())
cfg.update(domains=DOMAINS,conditions=CONDITIONS,seeds=SEEDS)
(RUN/'plan.json').write_text(json.dumps(cfg,indent=2))
print(cfg)

## Inspect the executed implementation
Finder selects one member and resets its Adam moments; only that latent and the active network are updated. Reader latents are constants. The code printed here is the code invoked below.

In [ ]:
print((RUN/'train.py').read_text())

## Train
Each subprocess prints fit losses. Checkpoints and per-batch/epoch loss arrays are written under runs/. Existing completed jobs are skipped on resume. Do not rerun the staging cell to resume.

In [ ]:
subprocess.run([sys.executable,str(RUN/'run_all.py'),'--execute'],cwd=RUN,check=True)

## Plot stored fit trajectories
This historical fixed-budget protocol has no validation selection; do not substitute a later validation-based model.

In [ ]:
import matplotlib.pyplot as plt
for d in DOMAINS:
    for seed in SEEDS:
        folder=RUN/'runs'/d/'direct'/f'seed_{seed}'/'finder'
        rows=[json.loads(p.read_text()) for p in sorted(folder.glob('phase_[0-9][0-9].json'))]
        for side in ['A','B']:
            q=[r for r in rows if r['side']==side]
            plt.plot([r['cycle'] for r in q],[r['fit_loss'][side] for r in q],label=f'{d} {seed} {side}')
plt.xlabel('Cycle');plt.ylabel('Fit data loss');plt.legend();plt.show()